In [ ]:
import numpy as np
from numpy import pi
from scipy import signal

# Especificaciones del Filtro ---

#frecuencia de corte real en Hz que queremos para el filtro
fP_real = 100;


# parametros del filtro
aP = .1     # rizado en la bamda de paso
aS = 25    # atenuación en la banda de rechazo


# debemos escalar la frecuencia porque la frecuencia normalizada del THAT es 1000


wP = 2*pi*fP_real/1000    # frecuencia de paso para 100 hz


factor = 2.1  # defino la banda de transicion como factor
wS = factor*wP   # frecuencia de rechazo normalizada

#  Determinación del Orden del Filtro (N)

N, Wn = signal.cheb1ord(wP, wS, aP, aS, analog=True)

print(f"Orden mínimo del filtro (N): {N}")
print(f"Frecuencia de corte angular (Wn) ajustada: {Wn:.4f} rad/s")
print()
# Diseño del Filtro Protoptipo y Coeficientes SOS ---

sos  = signal.cheby1(N, aP, Wn, btype='lowpass', analog=True, output='sos')

print("Estructura:")
print()

print("       b01                          1        ")
print("= -------------------   x    ------------------")
print("   s^2 + a11*s + a01          s^2 + a12*s + a02")
print()
print(f"         {sos[0][2]:0.3f}                            1        ")
print("= -----------------------   x    -----------------------")
print(f"   s^2 + {sos[0][4]:0.3f}*s + {sos[0][5]:0.3f}          s^2 + {sos[1][4]:0.3f}*s + {sos[1][5]:0.3f}")

print()
print("Ajuste en el computador analógico:")
print()
print(f"b_01 = {sos[0][2]:0.3f} => COEFF 1")
print(f"a_11 = {sos[0][4]:0.3f} => COEFF 2")
print(f"a_01 = {sos[0][5]:0.3f} => COEFF 3")
print(f"a_12 = {sos[1][4]:0.3f} => COEFF 4")
print(f"a_02 = {sos[1][5]:0.3f} => COEFF 5")
print()



# Respuesta en frecuencia real del filtro diseñado

In [ ]:
import matplotlib.pyplot as plt


# Convert SOS to ZPK for signal.bode

wP_real = 2*pi*fP_real/wP

b, a = signal.sos2tf(sos)

#  Convertimos el filtro a frecuencia real
sis = signal.lp2lp(b, a, wP_real)

#  Cálculo de la Respuesta en Frecuencia con Bode ---
# 2
print(wP_real)
w=np.logspace(np.log10(wP_real)-1 , np.log10(wP_real) +1)
w, mag, fase = signal.bode(sis, w, 500) # Increased points for smoother plot

f = w/(2*pi)

# --- 4. Gráficas de la Respuesta en Frecuencia ---
plt.figure(figsize=(10, 6))


plt.plot(f, mag, 'b')
plt.title('Respuesta en Frecuencia del Filtro Chebyshev Tipo I - Graficoecibeles')
plt.xlabel('Frecuencia Hz')
plt.ylabel('Ganancia (dB)')
plt.grid(True)
plt.axvline(fP_real, color='green', linestyle='--', label=f'Frecuencia de paso (fP={fP_real} Hz)')
plt.axvline(factor*fP_real, color='red', linestyle='--', label=f'Frecuencia de rechazo (fS={fP_real*factor} Hz)')
plt.axhline(-aP, color='purple', linestyle=':', label=f'Rizado en banda de paso (-{aP} dB)')
plt.axhline(-aS, color='orange', linestyle=':', label=f'Atenuación en banda de rechazo (-{aS} dB)')
plt.legend()
plt.xscale('log')
plt.ylim(-30, 5) # Ajustar límites para mejor visualización
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import pandas as pd

f =[0.5*fP_real, fP_real , 1.5*fP_real , factor*fP_real ]
w, mag, fase = signal.bode(sis, 2*pi*np.array(f)) # Increased points for smoother plot



respuesta = pd.DataFrame({
    'Frecuencia (Hz)': f,
    'Magnitud (dB)': mag,
    'Fase (grados)': fase
})


display(respuesta.round(1))